# Optional Ablation: Fine-Tuned DistilBERT vs. LLM Prompting

This notebook is the **one component in the whole project that needs a GPU** —
everything else (agents, RAG, calibration) runs fine on CPU. Run this on
Google Colab with a free GPU runtime (Runtime -> Change runtime type -> GPU).

**Purpose:** train a small, purpose-built classifier and compare its accuracy
against the LLM-based Content Agent's zero-shot judgments, on the same held-out
test set. This gives you a real "prompting vs. fine-tuning" ablation result for
the paper — it is optional and does not block the rest of the system.

In [ ]:
!pip install -q transformers datasets scikit-learn --break-system-packages

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load your labeled SMS/phishing text dataset here (NOT the RAG corpus --
# see data/README.md for the split rule). Replace with the real path.
df = pd.read_csv('../data/eval_sets/sample_eval_set.csv')
df['label_id'] = df['label'].map({'legitimate': 0, 'scam': 1})

train_df, test_df = train_test_split(df, test_size=0.2, stratify=df['label_id'], random_state=42)
print(f"Train: {len(train_df)}, Test: {len(test_df)}")

In [ ]:
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, Trainer, TrainingArguments
import torch

tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')
model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)

class ScamDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels):
        self.encodings = tokenizer(list(texts), truncation=True, padding=True, max_length=128)
        self.labels = list(labels)
    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.labels)

train_dataset = ScamDataset(train_df['text'], train_df['label_id'])
test_dataset = ScamDataset(test_df['text'], test_df['label_id'])

In [ ]:
training_args = TrainingArguments(
    output_dir='./distilbert_scam_classifier',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    optim='adamw_torch',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

trainer.train()

## Next: compare against the LLM Content Agent

Run `src/agents/content_agent.py`'s `run()` on the same `test_df` rows,
then compare precision/recall/F1 between this fine-tuned model and the
zero-shot LLM judgments. Report both in `docs/evaluation_plan.md`'s results
section once you have real numbers.